In [ ]:
%pip install pandas

KPI 2 Score par Copro :

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_base_copro.csv')
df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()

df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code', 'dept_nom']).agg({
    'lots_habitation': 'max', 
    'lots_parking': 'max',     
    'lat': 'first',
    'long': 'first'
}).reset_index()

df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)

df_final = df_unique[(df_unique['score_immeuble'] >= 0.5) & (df_unique['score_immeuble'] <= 0.9)].copy()
df_final['code_postal'] = df_final['code_postal'].fillna(0).astype(int).astype(str)

kpi2_score_immeuble = df_final[['code_postal', 'ville', 'adresse', 'lat', 'long', 'lots_habitation', 'score_immeuble']].copy()

kpi2_score_immeuble['score_immeuble'] = kpi2_score_immeuble['score_immeuble'].round(2)
kpi2_score_immeuble = kpi2_score_immeuble.sort_values('lots_habitation', ascending=False)
kpi2_score_immeuble = kpi2_score_immeuble.drop_duplicates(subset=['code_postal', 'lots_habitation'], keep='first')
kpi2_score_immeuble = kpi2_score_immeuble.reset_index(drop=True)

display(kpi2_score_immeuble.head(10))

Export des données pour l'équipe Front-End:

In [ ]:
kpi2_score_immeuble.to_csv('export_kpi2_immeubles.csv', index=False, encoding='utf-8')